In [1]:
#Social network

import networkx as nx
from collections import Counter
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Load the Excel file
file_path = "your_file_path_here.xlsx"  # Replace with your file path
data = pd.read_excel(file_path)

# Display the first few rows to inspect the data
print(data.head())

# Assuming the comments are in a column named 'comments'
comments = data['comments'].dropna()  # Remove missing or NaN values

# Inspect the first few comments
print(comments.head())

# Optional: Clean the comments (remove special characters, URLs, etc.)
import re

def clean_comment(comment):
    comment = re.sub(r"http\S+", "", comment)  # Remove URLs
    comment = re.sub(r"[^a-zA-Z\s]", "", comment)  # Remove special characters
    comment = comment.lower().strip()  # Convert to lowercase and strip whitespace
    return comment

comments = comments.apply(clean_comment)
print(comments.head())  # Check cleaned comments

# Convert comments to a list
comments_list = comments.tolist()

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(comments_list)
terms = vectorizer.get_feature_names_out()

# Build co-occurrence matrix and network as before
co_occurrence_matrix = (X.T @ X).toarray()



# Step 1: Tokenize and Build Co-occurrence Matrix
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(comments)
terms = vectorizer.get_feature_names_out()

# Co-occurrence matrix
co_occurrence_matrix = (X.T @ X).toarray()

# Step 2: Build Network
G = nx.Graph()
for i, term in enumerate(terms):
    for j in range(i + 1, len(terms)):
        if co_occurrence_matrix[i, j] > 0:
            G.add_edge(terms[i], terms[j], weight=co_occurrence_matrix[i, j])

# Step 3: Calculate Centrality
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)

print("Degree Centrality:", degree_centrality)
print("Betweenness Centrality:", betweenness_centrality)

AttributeError: 'CountVectorizer' object has no attribute 'get_feature_names_out'

In [2]:
#LDA

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 1: Vectorize Comments
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(comments)

# Step 2: Fit LDA Model
lda = LatentDirichletAllocation(n_components=2, random_state=42)
lda.fit(tfidf_matrix)

# Step 3: Print Topics
terms = tfidf_vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx}: {[terms[i] for i in topic.argsort()[:-10 - 1:-1]]}")

AttributeError: 'TfidfVectorizer' object has no attribute 'get_feature_names_out'

In [3]:
#Statistical testing

import scipy.stats as stats

# Example Centrality Scores for Testing
like_centrality = [degree_centrality.get('like', 0), betweenness_centrality.get('like', 0)]
love_centrality = [degree_centrality.get('love', 0), betweenness_centrality.get('love', 0)]

# t-test for Centrality Differences
t_stat, p_value = stats.ttest_ind(like_centrality, love_centrality)
print(f"T-test results: t-statistic = {t_stat}, p-value = {p_value}")

# Chi-Square for Topic Associations
like_topic_presence = [1, 0]  # Example: "like" occurs in Topic 1 but not Topic 2
love_topic_presence = [0, 1]  # Example: "love" occurs in Topic 2 but not Topic 1

chi2, p = stats.chisquare(f_obs=like_topic_presence, f_exp=love_topic_presence)
print(f"Chi-Square results: chi2 = {chi2}, p-value = {p}")

NameError: name 'degree_centrality' is not defined

In [4]:
#visualization
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
pos = nx.spring_layout(G, seed=42)  # Layout
nx.draw_networkx_nodes(G, pos, node_size=700, node_color='lightblue')
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=12, font_color='darkblue')
plt.title("Keyword Co-occurrence Network")
plt.show()

NameError: name 'G' is not defined

<Figure size 720x720 with 0 Axes>

In [6]:
# Focus on specific keywords like 'like' and 'love'
keywords_of_interest = ['like', 'love']

# Extract co-occurrences related to these keywords
edges_of_interest = [(term1, term2, weight) for term1, term2, weight in G.edges(data=True)
                     if term1 in keywords_of_interest or term2 in keywords_of_interest]

# Create a subgraph with only the edges of interest
subgraph = nx.Graph()
subgraph.add_edges_from((edge[0], edge[1], {'weight': edge[2]['weight']}) for edge in edges_of_interest)

# Visualize the subgraph
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
pos = nx.spring_layout(subgraph, seed=42)
nx.draw_networkx_nodes(subgraph, pos, node_size=500, node_color='lightblue')
nx.draw_networkx_edges(subgraph, pos, width=1.0, alpha=0.7)
nx.draw_networkx_labels(subgraph, pos, font_size=10, font_color='darkblue')
plt.title("Subgraph Centered on 'like' and 'love'")
plt.show()

# Calculate centralities for the subgraph
degree_centrality = nx.degree_centrality(subgraph)
print("Degree Centrality (Subgraph):", degree_centrality)

NameError: name 'G' is not defined

In [7]:
# Filter comments mentioning 'like' or 'love'
filtered_comments = comments[comments.str.contains(r'\blike\b|\blove\b', case=False)]
filtered_comments_list = filtered_comments.tolist()

AttributeError: 'list' object has no attribute 'str'

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Vectorize filtered comments
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(filtered_comments_list)

# Fit LDA
lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(tfidf_matrix)

# Display topics
terms = tfidf_vectorizer.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    print(f"Topic {idx}: {[terms[i] for i in topic.argsort()[:-10 - 1:-1]]}")

NameError: name 'filtered_comments_list' is not defined

In [9]:
import scipy.stats as stats

# Example centrality scores for "like" and "love"
like_centrality = degree_centrality.get('like', 0)
love_centrality = degree_centrality.get('love', 0)

# Perform a t-test
t_stat, p_value = stats.ttest_1samp([like_centrality, love_centrality], 0)
print(f"T-test results: t-statistic = {t_stat}, p-value = {p_value}")

NameError: name 'degree_centrality' is not defined

In [10]:
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.feature_extraction.text import CountVectorizer

# Sample comments (replace this with your cleaned comments list)
comments = [
    "I like this destination, it's very informative.",
    "I love the scenery! Absolutely breathtaking.",
    "This is a fantastic video, love it!",
    "Great tips, I really like the explanations here."
]

# Step 1: Build the Co-occurrence Network
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(comments)
terms = vectorizer.get_feature_names_out()

# Build co-occurrence matrix
co_occurrence_matrix = (X.T @ X).toarray()

# Create a graph
G = nx.Graph()
for i, term in enumerate(terms):
    for j in range(i + 1, len(terms)):
        if co_occurrence_matrix[i, j] > 0:
            G.add_edge(terms[i], terms[j], weight=co_occurrence_matrix[i, j])

# Step 2: Calculate Node Sizes Based on Frequency
term_frequency = X.toarray().sum(axis=0)  # Frequency of terms
node_sizes = [term_frequency[terms.tolist().index(node)] * 100 for node in G.nodes]

# Step 3: Visualize the Network with Node Sizes
plt.figure(figsize=(10, 10))
pos = nx.spring_layout(G, seed=42)  # Layout
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightblue', alpha=0.8)
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=10, font_color='darkblue')
plt.title("Keyword Co-occurrence Network with Node Sizes")
plt.show()

AttributeError: 'CountVectorizer' object has no attribute 'get_feature_names_out'

In [11]:
# Customize sizes for specific keywords
special_keywords = ['like', 'love']
node_colors = ['red' if node in special_keywords else 'lightblue' for node in G.nodes]
node_sizes = [term_frequency[terms.tolist().index(node)] * 200 if node in special_keywords else term_frequency[terms.tolist().index(node)] * 100 for node in G.nodes]

# Visualize
plt.figure(figsize=(10, 10))
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8)
nx.draw_networkx_edges(G, pos, width=1.0, alpha=0.5)
nx.draw_networkx_labels(G, pos, font_size=10, font_color='darkblue')
plt.title("Keyword Co-occurrence Network Highlighting 'like' and 'love'")
plt.show()

NameError: name 'G' is not defined